# False-positive classifier — training

Ranks findings by how likely the author is to accept them, learned from your own
accept/dismiss decisions. `plan.md` §9 calls this the moat: the label set is yours and
nobody else has it.

**Input:** `*_labels.jsonl` files exported from the app (workspace bar → *Export training
labels*). One row per decided finding, produced by `src/lib/export/trainingData.ts`.

**Don't run this yet if you have < ~300 labelled rows.** LightGBM on a few dozen examples
will memorise noise and rank worse than the existing rules. The honest move is to keep
using the app and exporting until the count is there.

Upgrade path from `plan.md`: LightGBM now → fine-tuned DeBERTa-v3-small at ~2k labels.

In [ ]:
!pip -q install lightgbm scikit-learn pandas matplotlib

import glob, json
import numpy as np, pandas as pd

paths = sorted(glob.glob('/kaggle/input/**/*_labels.jsonl', recursive=True)) or sorted(glob.glob('*_labels.jsonl'))
rows = [json.loads(l) for p in paths for l in open(p, encoding='utf-8') if l.strip()]
df = pd.DataFrame(rows)
print(f'{len(df)} labelled findings from {len(paths)} export(s)')

MIN_ROWS = 300
if len(df) < MIN_ROWS:
    print(f'\n*** Only {len(df)} rows. Below ~{MIN_ROWS} this model will underperform the rules.')
    print('*** Keep using the app and exporting labels; re-run when the count is there.')

display(df[['kind', 'category', 'label']].value_counts().head(20))
print('\naccept rate:', df['label'].mean().round(3))

## Features

The `f_*` columns come straight from the exporter. Free-text is deliberately excluded from
the tabular model — with a few hundred rows it would overfit to the vocabulary of whichever
papers you happened to analyse.

In [ ]:
feature_cols = [c for c in df.columns if c.startswith('f_')]
cat_cols = ['kind', 'category', 'f_source_type', 'f_section']

X = df[feature_cols + [c for c in cat_cols if c not in feature_cols]].copy()
for c in X.columns:
    if X[c].dtype == object:
        X[c] = X[c].astype('category')
y = df['label'].values

print(X.dtypes)
print('\nmissing per column:\n', X.isna().mean().round(3).sort_values(ascending=False).head())

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# Cross-validated: with a small label set a single holdout is mostly noise
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(df))
models = []

params = dict(objective='binary', learning_rate=0.05, num_leaves=15,
              min_child_samples=10, feature_fraction=0.8, bagging_fraction=0.8,
              bagging_freq=1, verbose=-1, seed=42)

for tr, va in skf.split(X, y):
    m = lgb.train(params, lgb.Dataset(X.iloc[tr], y[tr]),
                  num_boost_round=400,
                  valid_sets=[lgb.Dataset(X.iloc[va], y[va])],
                  callbacks=[lgb.early_stopping(40, verbose=False)])
    oof[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
    models.append(m)

print('OOF ROC-AUC:', round(roc_auc_score(y, oof), 3))
print('OOF PR-AUC :', round(average_precision_score(y, oof), 3))
print()
print(classification_report(y, (oof >= 0.5).astype(int), digits=3))

### Does it beat the rules?

The comparison that decides whether to ship: the existing `confidence` score is already a
ranking. If the model doesn't clearly beat it, keep the rules — they're free, debuggable,
and don't drift.

In [ ]:
baseline = df['f_confidence'].fillna(0.5).values
b_auc, m_auc = roc_auc_score(y, baseline), roc_auc_score(y, oof)
print(f'rules confidence ROC-AUC : {b_auc:.3f}')
print(f'LightGBM        ROC-AUC : {m_auc:.3f}')
print(f'\ndelta: {m_auc - b_auc:+.3f}')
print('SHIP' if m_auc > b_auc + 0.03 else 'KEEP THE RULES — not a convincing margin yet')

In [ ]:
import matplotlib.pyplot as plt

imp = pd.DataFrame({'feature': models[0].feature_name(),
                    'gain': np.mean([m.feature_importance('gain') for m in models], axis=0)})
imp = imp.sort_values('gain', ascending=False).head(20)
plt.figure(figsize=(7, 6))
plt.barh(imp['feature'][::-1], imp['gain'][::-1])
plt.title('Feature importance (mean gain)')
plt.tight_layout(); plt.show()

# Which categories does the author dismiss most? Often more actionable than the model —
# a category dismissed 90% of the time usually means a rule needs fixing, not reweighting.
display(df.groupby('category')['label'].agg(['mean', 'count']).sort_values('mean'))

## Export

LightGBM has no browser runtime, so dump the trees as JSON and evaluate them in TypeScript
(a few hundred lines) — or keep this server-side in the Netlify function. Either way the
paper text still never leaves the machine: only the `f_*` feature vector is scored.

In [ ]:
best = models[int(np.argmax([roc_auc_score(y[va], oof[va]) for _, va in skf.split(X, y)]))]
best.save_model('fp_classifier.txt')
json.dump(best.dump_model(), open('fp_classifier.json', 'w'))
json.dump({'oof_roc_auc': float(m_auc), 'baseline_roc_auc': float(b_auc),
           'n_labels': int(len(df)), 'features': list(X.columns)},
          open('fp_metrics.json', 'w'), indent=2)
print('saved fp_classifier.{txt,json} + fp_metrics.json')